# Hyperparameter Optimization - Exercises

10 exercises covering configuration spaces, grid/random search, log-scale sampling,
Bayesian optimization, successive halving, Hyperband, validation bias, Pareto frontiers,
and LLM fine-tuning study design.

| Format | Description |
| --- | --- |
| Problem | Markdown cell with the task |
| Your Solution | Runnable scaffold with placeholders |
| Solution | Reference solution with checks |

In [ ]:
import math
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style="whitegrid", palette="colorblind")
        HAS_SNS = True
    except ImportError:
        HAS_SNS = False
    mpl.rcParams.update({
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "legend.framealpha": 0.85,
        "lines.linewidth": 2.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "savefig.dpi": 150,
    })
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    HAS_SNS = False

COLORS = {
    "primary": "#0077BB",
    "secondary": "#EE7733",
    "tertiary": "#009988",
    "error": "#CC3311",
    "neutral": "#555555",
    "highlight": "#EE3377",
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def check_true(name, condition):
    ok = bool(condition)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    assert ok, name

def check_close(name, value, target, tol=1e-8):
    ok = abs(float(value) - float(target)) <= tol
    print(f"{'PASS' if ok else 'FAIL'} - {name}: {value} vs {target}")
    assert ok, name

def normal_pdf(z):
    z = np.asarray(z, dtype=float)
    return np.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)

def normal_cdf(z):
    z = np.asarray(z, dtype=float)
    erf_vec = np.vectorize(math.erf)
    return 0.5 * (1.0 + erf_vec(z / math.sqrt(2.0)))

def synthetic_hpo_objective(lr, weight_decay, rank=8, noise=0.0, seed=None):
    rng = np.random.default_rng(seed)
    x = math.log10(lr)
    wd = math.log10(weight_decay + 1e-8)
    rank_penalty = 0.002 * (rank - 16) ** 2 / 16.0
    bowl = (x + 3.7) ** 2 + 0.18 * (wd + 2.2) ** 2 + rank_penalty
    interaction = 0.08 * math.sin(4.0 * x + 1.5 * wd)
    return 0.35 + bowl + interaction + rng.normal(0.0, noise)

def sample_log_uniform(low_exp, high_exp, n, rng):
    return 10 ** rng.uniform(low_exp, high_exp, size=n)

def rbf_kernel(X1, X2, lengthscale=0.25, variance=1.0):
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    sq = np.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=2)
    return variance * np.exp(-0.5 * sq / (lengthscale ** 2))

def gp_predict(X_train, y_train, X_test, lengthscale=0.25, noise=1e-6):
    X_train = np.atleast_2d(X_train)
    X_test = np.atleast_2d(X_test)
    y_train = np.asarray(y_train)
    K = rbf_kernel(X_train, X_train, lengthscale) + noise * np.eye(len(X_train))
    Ks = rbf_kernel(X_train, X_test, lengthscale)
    Kss = rbf_kernel(X_test, X_test, lengthscale)
    L = la.cholesky(K)
    alpha = la.solve(L.T, la.solve(L, y_train))
    mu = Ks.T @ alpha
    v = la.solve(L, Ks)
    cov = Kss - v.T @ v
    var = np.maximum(np.diag(cov), 1e-12)
    return mu, var

def expected_improvement(mu, sigma, best, xi=0.0):
    sigma = np.maximum(np.asarray(sigma), 1e-12)
    z = (best - xi - np.asarray(mu)) / sigma
    return (best - xi - np.asarray(mu)) * normal_cdf(z) + sigma * normal_pdf(z)

print("Setup complete.")
print("NumPy version:", np.__version__)
print("Matplotlib available:", HAS_MPL)

    ---

    ## Exercise 1 * - Classify HPO objects

    Classify each item as a model parameter, hyperparameter, metric, resource, or metadata:

(a) Transformer weight matrix $W$.
(b) Learning rate $\eta$.
(c) Validation loss after 2000 steps.
(d) Number of epochs.
(e) Git commit hash.

In [ ]:
# Your Solution - Exercise 1: Classify HPO objects
answers = {
    "W": None,
    "learning_rate": None,
    "validation_loss": None,
    "epochs": None,
    "git_commit": None,
}
print(answers)

In [ ]:
header("Exercise 1: Classify HPO objects")
answers = {
    "W": "model parameter",
    "learning_rate": "hyperparameter",
    "validation_loss": "metric",
    "epochs": "resource",
    "git_commit": "metadata",
}
for key, value in answers.items():
    print(f"{key:16s}: {value}")
check_true("learning rate is a hyperparameter", answers["learning_rate"] == "hyperparameter")
print("\nTakeaway: HPO optimizes choices around training, not the learned weights themselves.")

    ---

    ## Exercise 2 * - Grid size and cost

    A grid has 4 learning rates, 3 weight decays, 4 LoRA ranks, and 2 batch sizes.
Each trial costs 45 minutes.
Compute trial count and GPU-hours, then add two more 5-value hyperparameters.

In [ ]:
# Your Solution - Exercise 2: Grid size and cost
trial_count = None
gpu_hours = None
expanded_count = None
print(trial_count, gpu_hours, expanded_count)

In [ ]:
header("Exercise 2: Grid size and cost")
trial_count = 4 * 3 * 4 * 2
gpu_hours = trial_count * 45 / 60
expanded_count = trial_count * 5 * 5
print("trial count:", trial_count)
print("GPU-hours:", gpu_hours)
print("expanded count:", expanded_count)
check_close("trial count", trial_count, 96)
check_close("GPU-hours", gpu_hours, 72.0)
check_close("expanded grid", expanded_count, 2400)
print("\nTakeaway: grid cost grows multiplicatively with every added coordinate.")

    ---

    ## Exercise 3 * - Random-search hit probability

    If a good region has probability $q(\mathcal{A})=0.03$ under the search distribution,
compute the probability of at least one hit after $N=20,50,100$ trials and the $N$ needed for 95 percent.

In [ ]:
# Your Solution - Exercise 3: Random-search hit probability
q_good = 0.03
probabilities = None
n_for_95 = None
print(probabilities, n_for_95)

In [ ]:
header("Exercise 3: Random-search hit probability")
q_good = 0.03
Ns = np.array([20, 50, 100])
probabilities = 1 - (1 - q_good) ** Ns
n_for_95 = math.ceil(math.log(1 - 0.95) / math.log(1 - q_good))
print("probabilities:", dict(zip(Ns, np.round(probabilities, 4))))
print("N for 95%:", n_for_95)
check_true("100 trials has higher hit probability than 20", probabilities[-1] > probabilities[0])
check_true("N for 95 is positive", n_for_95 > 0)
print("\nTakeaway: random search is strong when the search distribution gives good regions enough mass.")

    ---

    ## Exercise 4 ** - Log-uniform learning rates

    Let $\log_{10}\eta \sim \mathcal{U}(-5,-2)$.
Compute $P(10^{-4} \le \eta \le 10^{-3})$ and compare with linear-uniform sampling over $[10^{-5},10^{-2}]$.

In [ ]:
# Your Solution - Exercise 4: Log-uniform learning rates
p_log_uniform = None
p_linear_uniform = None
print(p_log_uniform, p_linear_uniform)

In [ ]:
header("Exercise 4: Log-uniform learning rates")
p_log_uniform = ( -3 - (-4) ) / ( -2 - (-5) )
p_linear_uniform = (1e-3 - 1e-4) / (1e-2 - 1e-5)
print("log-uniform probability:", p_log_uniform)
print("linear-uniform probability:", p_linear_uniform)
check_close("log-uniform mass", p_log_uniform, 1/3)
check_true("log-uniform gives more mass to the decade", p_log_uniform > p_linear_uniform)
print("\nTakeaway: log-uniform sampling treats multiplicative decades evenly.")

    ---

    ## Exercise 5 ** - Expected improvement

    For $F(\boldsymbol{\lambda}) \sim \mathcal{N}(\mu,\sigma^2)$ with $\mu=0.42$, $\sigma=0.05$,
current best $y_{\min}=0.40$, and $\xi=0$, compute $z$, PI, and EI.

In [ ]:
# Your Solution - Exercise 5: Expected improvement
mu, sigma, best = 0.42, 0.05, 0.40
z = None
pi = None
ei = None
print(z, pi, ei)

In [ ]:
header("Exercise 5: Expected improvement")
mu, sigma, best = 0.42, 0.05, 0.40
z = (best - mu) / sigma
pi = float(normal_cdf(z))
ei = float(expected_improvement(mu, sigma, best))
print("z:", z)
print("PI:", pi)
print("EI:", ei)
check_true("PI is between 0 and 1", 0 <= pi <= 1)
check_true("EI is nonnegative", ei >= 0)
print("\nTakeaway: EI can be positive even when the mean is worse than the incumbent because uncertainty has value.")

    ---

    ## Exercise 6 ** - Successive halving budget

    Start with 27 configurations, reduction factor 3, and initial resource 1.
Compute candidates, resources, and total budget across rungs.

In [ ]:
# Your Solution - Exercise 6: Successive halving budget
candidates = None
resources = None
total_budget = None
print(candidates, resources, total_budget)

In [ ]:
header("Exercise 6: Successive halving budget")
candidates = [27, 9, 3, 1]
resources = [1, 3, 9, 27]
total_budget = sum(c * r for c, r in zip(candidates, resources))
full_budget = 27 * 27
print("candidates:", candidates)
print("resources:", resources)
print("successive-halving budget:", total_budget)
print("full budget:", full_budget)
check_close("SH budget", total_budget, 108)
check_true("SH is cheaper than full training", total_budget < full_budget)
print("\nTakeaway: successive halving trades broad cheap exploration for narrow expensive confirmation.")

    ---

    ## Exercise 7 ** - Hyperband brackets

    Let $R=81$ and $\eta=3$.
Compute $s_{\max}$ and list bracket starting sizes and initial resources.

In [ ]:
# Your Solution - Exercise 7: Hyperband brackets
R, eta = 81, 3
s_max = None
brackets = None
print(s_max, brackets)

In [ ]:
header("Exercise 7: Hyperband brackets")
R, eta = 81, 3
s_max = int(math.floor(math.log(R, eta)))
brackets = []
for s in range(s_max, -1, -1):
    n = math.ceil((s_max + 1) / (s + 1) * eta ** s)
    r = R * eta ** (-s)
    brackets.append((s, n, r))
print("s_max:", s_max)
print("brackets:", brackets)
check_close("s_max", s_max, 4)
check_true("multiple brackets hedge assumptions", len(brackets) == s_max + 1)
print("\nTakeaway: Hyperband hedges between many-short and few-long training strategies.")

    ---

    ## Exercise 8 *** - Selection bias simulation

    Simulate 100 equal-quality configurations with observed scores $Y_j \sim \mathcal{N}(0.5,0.02^2)$.
Estimate the expected best observed score over many repetitions.

In [ ]:
# Your Solution - Exercise 8: Selection bias simulation
expected_best = None
optimism = None
print(expected_best, optimism)

In [ ]:
header("Exercise 8: Selection bias simulation")
rng = np.random.default_rng(888)
reps = 5000
observed_best = []
for _ in range(reps):
    scores = 0.5 + rng.normal(0, 0.02, size=100)
    observed_best.append(scores.min())
expected_best = float(np.mean(observed_best))
optimism = 0.5 - expected_best
print("expected best observed score:", expected_best)
print("optimism:", optimism)
check_true("best observed score is optimistic", expected_best < 0.5)
check_true("optimism is meaningful", optimism > 0.03)
print("\nTakeaway: selecting the minimum of many noisy scores creates an optimistic winner.")

---

## Exercise 9 *** - Pareto frontier

Given configurations A-E with loss and latency, find dominated points and the Pareto frontier.

In [ ]:
# Your Solution - Exercise 9: Pareto frontier
names = np.array(["A", "B", "C", "D", "E"])
points = np.array([[0.42, 120], [0.40, 180], [0.45, 90], [0.41, 110], [0.39, 240]])
frontier = None
print(frontier)

In [ ]:
header("Exercise 9: Pareto frontier")
names = np.array(["A", "B", "C", "D", "E"])
points = np.array([[0.42, 120], [0.40, 180], [0.45, 90], [0.41, 110], [0.39, 240]], dtype=float)
def dominated(i):
    return any(np.all(points[j] <= points[i]) and np.any(points[j] < points[i]) for j in range(len(points)) if j != i)
dominated_names = [names[i] for i in range(len(points)) if dominated(i)]
frontier = [names[i] for i in range(len(points)) if not dominated(i)]
feasible = [i for i in range(len(points)) if points[i, 1] <= 130]
best_feasible = names[min(feasible, key=lambda i: points[i, 0])]
print("dominated:", dominated_names)
print("frontier:", frontier)
print("best feasible latency <= 130:", best_feasible)
check_true("A is dominated", "A" in dominated_names)
check_true("D is frontier", "D" in frontier)
print("\nTakeaway: multi-objective HPO should report tradeoffs, not only one scalar winner.")

    ---

    ## Exercise 10 *** - LLM fine-tuning study design

    Design a 60-trial LoRA fine-tuning HPO study.
Specify search space, metric, pruning rule, seed policy, and final selection.

In [ ]:
# Your Solution - Exercise 10: LLM fine-tuning study design
study = {
    "search_space": None,
    "metric": None,
    "budget": None,
    "seed_policy": None,
    "selection_rule": None,
}
print(study)

In [ ]:
header("Exercise 10: LLM fine-tuning study design")
study = {
    "search_space": {
        "lr": "log-uniform(1e-5, 1e-3)",
        "weight_decay": [0.0, 0.01, 0.05],
        "rank": [4, 8, 16, 32],
        "warmup_ratio": "uniform(0.0, 0.1)",
        "target_modules": ["qv", "qkvo", "all_linear"],
    },
    "metric": "validation negative log-likelihood with task accuracy guardrail",
    "budget": "60 trials, max 3000 steps, ASHA grace period 500",
    "seed_policy": "one seed for exploration, five seeds for top three candidates",
    "selection_rule": "lowest repeated mean validation loss subject to guardrails, then final untouched test",
}
for key, value in study.items():
    print(key, ":", value)
check_true("budget mentions 60 trials", "60" in study["budget"])
check_true("final selection uses untouched test", "test" in study["selection_rule"])
print("\nTakeaway: a good HPO study specifies the experiment, not just the tuner.")